# MIRA on PhysioNet 2012: zero-shot forecasting with the published checkpoint in PyHealth

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/sunlabuiuc/PyHealth/blob/master/examples/mira_physionet2012_forecasting.ipynb)

This notebook runs

> Hao Li, Bowen Deng, Chang Xu, Zhiyuan Feng, Viktor Schlegel, Yu-Hao Huang, Yizheng Sun, Jingyuan Sun, Kailai Yang, Yiyao Yu, Jiang Bian.
> **MIRA: Medical Time Series Foundation Model for Real-World Health Data.** NeurIPS 2025.
> [arXiv:2506.07584](https://arxiv.org/abs/2506.07584) · [authors' code](https://github.com/microsoft/MIRA) · [checkpoint](https://huggingface.co/MIRA-Mode/MIRA)

with PyHealth's `MIRA` model, which loads the published MIRA-Large checkpoint (455M parameters, MIT License) with every
weight matched. MIRA is a decoder-only transformer for irregularly sampled medical series: continuous-time rotary
positions (CT-RoPE), a mixture of experts and a neural-ODE extrapolation block.

**What is reproduced.** The paper's only open, inherently irregular benchmark is CinC 2012 (PhysioNet 2012), where it
reports zero-shot MIRA forecasts (Table 2). The paper does not describe that benchmark's windows or normalization, so we
use PyHealth's PhysioNet 2012 forecasting benchmark instead, the one the APN notebook
(`examples/apn_physionet2012_forecasting.ipynb`) reproduces: from the first 36 hours of an ICU stay, forecast the 36
vital signs and labs observed at the next 3 observation times; hourly means, z-scored, test records = last 10% by
`RecordID`. MIRA runs **zero-shot** (no training), next to APN trained on the same split and last observation carried
forward (LOCF).

**Paper's numbers on CinC 2012** (Table 2, zero-shot, labelled "(10¹)", averaged over prediction lengths):
MIRA-Large RMSE **6.221**, MAE **6.115**; best full-shot baseline ContiFormer RMSE 5.987, MAE 5.985.

MIRA-Large is compute-heavy: forecasting the 1,192 test records takes about 16 minutes on a laptop CPU (about 25 s per
batch of 32 records), so a GPU runtime is recommended.

## 1. Install PyHealth

MIRA is on PyHealth's master branch once its pull request is merged, and will be in the next PyPI release. Until then, install from GitHub.

In [1]:
!pip install -q "git+https://github.com/sunlabuiuc/PyHealth"
# Optional: the ODE block (use_ode=True) uses torchdiffeq when it is installed,
# as the authors' code does; without it, a fixed-step RK4 solver gives slightly
# different numbers.
# !pip install -q torchdiffeq

## 2. Download PhysioNet 2012, build the task and the benchmark split

The same steps as the APN notebook, so all three methods are scored on the same 1,192 test records.

In [2]:
import numpy as np
import pandas as pd
import torch

from pyhealth.datasets import PhysioNet2012Dataset, get_dataloader
from pyhealth.models import APN, MIRA
from pyhealth.tasks import (
    PhysioNet2012Forecasting,
    physionet2012_benchmark_split,
    physionet2012_statistics,
)

ROOT = "physionet2012"          # raw files and PyHealth's tables go here
CACHE = "physionet2012_cache"   # PyHealth's processed cache
if torch.cuda.is_available():
    DEVICE = "cuda"
elif torch.backends.mps.is_available():
    DEVICE = "mps"
else:
    DEVICE = "cpu"
print("device:", DEVICE)

dataset = PhysioNet2012Dataset(root=ROOT, download=True, cache_dir=CACHE, num_workers=2)
mean, std = physionet2012_statistics(dataset)
samples = dataset.set_task(PhysioNet2012Forecasting(mean=mean, std=std), num_workers=2)
train, val, test = physionet2012_benchmark_split(samples)
print(f"{len(samples)} records: {len(train)} train, {len(val)} validation, {len(test)} test")

device: cpu
Initializing physionet2012 dataset from physionet2012 (dev mode: False)


Using provided cache_dir: physionet2012_cache/ebfeb3a0-15d4-548f-bbd3-ea3304350207


Found cached event dataframe: physionet2012_cache/ebfeb3a0-15d4-548f-bbd3-ea3304350207/global_event_df.parquet


Setting task PhysioNet2012Forecasting for physionet2012 base dataset...


Task cache paths: task_df=physionet2012_cache/ebfeb3a0-15d4-548f-bbd3-ea3304350207/tasks/PhysioNet2012Forecasting_e32c09bb-9d83-5030-9b0f-106ee24fad72/task_df.ld, samples=physionet2012_cache/ebfeb3a0-15d4-548f-bbd3-ea3304350207/tasks/PhysioNet2012Forecasting_e32c09bb-9d83-5030-9b0f-106ee24fad72/samples_8d49b860-4994-5238-8a90-156b27a35d31.ld


Found cached processed samples at physionet2012_cache/ebfeb3a0-15d4-548f-bbd3-ea3304350207/tasks/PhysioNet2012Forecasting_e32c09bb-9d83-5030-9b0f-106ee24fad72/samples_8d49b860-4994-5238-8a90-156b27a35d31.ld, skipping processing.


11917 records: 9652 train, 1073 validation, 1192 test


## 3. Scoring

Two ways of averaging, both over the observed test targets in z-score units:

- **pooled** (the benchmark's and APN's): MSE and MAE over all observed targets; we also show RMSE = √MSE;
- **per series** (closer to the paper's "averaged across evaluation windows" and to `model_eval.py` in the MIRA repo):
  the RMSE and MAE of each (record, variable) series with an observed target, then averaged.

In [3]:
def score(predict, loader):
    se = ae = n = 0.0
    series_rmse, series_mae = [], []
    for batch in loader:
        y = batch["y"].to(DEVICE)
        with torch.no_grad():
            pred = predict(batch).to(DEVICE)
        mask = torch.isfinite(y)
        d = torch.where(mask, pred - torch.nan_to_num(y), 0.0)
        se += float((d ** 2).sum()); ae += float(d.abs().sum()); n += int(mask.sum())
        count = mask.sum(dim=1)                      # [B, N] observed steps per series
        has = count > 0
        series_rmse.append(((d ** 2).sum(1)[has] / count[has]).sqrt().cpu())
        series_mae.append((d.abs().sum(1)[has] / count[has]).cpu())
    return {
        "MSE": se / n, "RMSE": (se / n) ** 0.5, "MAE": ae / n,
        "series RMSE": float(torch.cat(series_rmse).mean()),
        "series MAE": float(torch.cat(series_mae).mean()),
    }

def locf(batch):
    x = batch["x"].to(DEVICE)
    last = torch.zeros(x.shape[0], x.shape[2], device=DEVICE)
    for t in range(x.shape[1]):
        last = torch.where(torch.isfinite(x[:, t]), x[:, t], last)
    return last.unsqueeze(1).expand(-1, batch["y"].shape[1], -1)

test_loader = get_dataloader(test, batch_size=32)
results = {"Last observation carried forward": score(locf, test_loader)}
results["Last observation carried forward"]

{'MSE': 0.41629158714211667,
 'RMSE': 0.6452066236037233,
 'MAE': 0.4115064246923365,
 'series RMSE': 0.403878778219223,
 'series MAE': 0.3731636703014374}

## 4. MIRA zero-shot with the published checkpoint

`MIRA(dataset, pretrained=True)` downloads MIRA-Large from Hugging Face (about 0.9 GB, pinned to a commit) and raises an
error unless every weight of the checkpoint matches the model. Each variable is forecast on its own: its observed
hourly values in the first 36 hours (an irregular series), standardized with their own mean and standard deviation,
are forecast autoregressively at the 3 target times. A variable that was never observed is forecast as 0 (the mean).

By default the neural-ODE block is off, as in the authors' inference example (`next_target_time_values=None`). The
second run turns it on, integrating the last hidden state to each target time (times in hours).

In [4]:
mira = MIRA(test, pretrained=True).to(DEVICE).eval()
print(f"MIRA-Large: {sum(p.numel() for p in mira.parameters()):,} parameters, all loaded from the checkpoint")

results["MIRA-Large zero-shot (PyHealth)"] = score(lambda b: mira(**b)["forecast"], test_loader)

mira.use_ode, mira.ode_time_scale = True, 48.0   # PhysioNet2012Forecasting times are hour / 48
results["MIRA-Large zero-shot + ODE block (PyHealth)"] = score(lambda b: mira(**b)["forecast"], test_loader)
mira.use_ode = False
pd.DataFrame(results).T.round(4)

MIRA-Large: 453,331,072 parameters, all loaded from the checkpoint


,MSE,RMSE,MAE,series RMSE,series MAE
Last observation carried forward,0.4163,0.6452,0.4115,0.4039,0.3732
MIRA-Large zero-shot (PyHealth),0.3341,0.5780,0.3881,0.3945,0.3654
MIRA-Large zero-shot + ODE block (PyHealth),0.3347,0.5785,0.3892,0.3973,0.3680


## 5. APN on the same split

APN trained with its paper's protocol (as in the APN notebook; one seed here, `SEEDS = [0, 1, 2, 3, 4]` for five).
On a CPU one run takes several minutes.

In [5]:
import copy

SEEDS = [0]
APN_DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

def train_apn(seed, max_epochs=200, patience=10):
    torch.manual_seed(seed); np.random.seed(seed)
    model = APN(train, hidden_dim=24, num_patches=20, time_dim=8, dropout=0.1).to(APN_DEVICE)
    opt = torch.optim.Adam(model.parameters(), lr=0.03)
    sched = torch.optim.lr_scheduler.LambdaLR(opt, lambda e: 1.0 if e < 2 else 0.8 ** (e - 2))
    train_loader = get_dataloader(train, batch_size=32, shuffle=True)
    val_loader = get_dataloader(val, batch_size=256)
    best, best_state, wait = float("inf"), None, 0
    for epoch in range(max_epochs):
        model.train()
        for batch in train_loader:
            opt.zero_grad()
            model(**batch)["loss"].backward()
            opt.step()
        sched.step()
        model.eval()
        val_mse = score(lambda b: model(**b)["forecast"], val_loader)["MSE"]
        if val_mse < best:
            best, best_state, wait = val_mse, copy.deepcopy(model.state_dict()), 0
        else:
            wait += 1
            if wait >= patience:
                break
    model.load_state_dict(best_state)
    print(f"seed {seed}: stopped after epoch {epoch}, best validation MSE {best:.4f}")
    return model.eval()

apn_runs = []
for seed in SEEDS:
    apn = train_apn(seed)
    apn_runs.append(score(lambda b: apn(**b)["forecast"], test_loader))
results[f"APN, trained ({len(SEEDS)} seed{'s' if len(SEEDS) > 1 else ''}, PyHealth)"] = (
    pd.DataFrame(apn_runs).mean().to_dict()
)

seed 0: stopped after epoch 27, best validation MSE 0.3135


## 6. Results next to the papers

In [6]:
ours = pd.DataFrame(results).T
paper = pd.DataFrame(
    {
        "MIRA-Large zero-shot, MIRA paper (CinC 2012, '(10^1)')": {"RMSE": 6.221, "MAE": 6.115},
        "ContiFormer full-shot, MIRA paper (CinC 2012, '(10^1)')": {"RMSE": 5.987, "MAE": 5.985},
        "T-PatchGNN full-shot, MIRA paper (CinC 2012, '(10^1)')": {"RMSE": 6.247, "MAE": 6.246},
        "APN, APN paper (PhysioNet 2012, 5 runs)": {"MSE": 0.3093, "RMSE": 0.3093 ** 0.5, "MAE": 0.3650},
    }
).T
pd.concat([ours, paper]).round(4)

,MSE,RMSE,MAE,series RMSE,series MAE
Last observation carried forward,0.4163,0.6452,0.4115,0.4039,0.3732
MIRA-Large zero-shot (PyHealth),0.3341,0.5780,0.3881,0.3945,0.3654
MIRA-Large zero-shot + ODE block (PyHealth),0.3347,0.5785,0.3892,0.3973,0.3680
"APN, trained (1 seed, PyHealth)",0.3050,0.5523,0.3706,0.3697,0.3422
"MIRA-Large zero-shot, MIRA paper (CinC 2012, '(10^1)')",NaN,6.2210,6.1150,NaN,NaN
"ContiFormer full-shot, MIRA paper (CinC 2012, '(10^1)')",NaN,5.9870,5.9850,NaN,NaN
"T-PatchGNN full-shot, MIRA paper (CinC 2012, '(10^1)')",NaN,6.2470,6.2460,NaN,NaN
"APN, APN paper (PhysioNet 2012, 5 runs)",0.3093,0.5561,0.3650,NaN,NaN


### Our run (local CPU, one APN seed)

| Model | Test MSE | RMSE | MAE |
|---|---|---|---|
| Last observation carried forward | 0.4163 | 0.6452 | 0.4115 |
| **MIRA-Large zero-shot** (no training) | **0.3341** | **0.5780** | **0.3881** |
| MIRA-Large zero-shot + ODE block | 0.3347 | 0.5785 | 0.3892 |
| APN, trained on this split (seed 0) | 0.3050 | 0.5523 | 0.3706 |
| APN, APN paper (5 runs) | 0.3093 | 0.5561 | 0.3650 |
| MIRA-Large, MIRA paper (CinC 2012, "(10¹)") | – | 6.221 | 6.115 |

Without any training, MIRA-Large cuts the MSE of last observation carried forward by 20% (0.416 to 0.334), but it
stays behind APN trained on this split (0.305), in line with the paper, where full-shot models such as ContiFormer
beat zero-shot MIRA on CinC 2012. The ODE block changes little here. If the paper's "(10¹)" means its values are
×10, its MIRA-Large RMSE (0.622) is close to ours (0.578), but the setups differ (see below).

## 7. How our setup differs from the paper's

- **Checkpoint.** The Hugging Face model card describes `MIRA-Mode/MIRA` as the large configuration of the official
  release, "trained exclusively on publicly available time series datasets", and warns that downstream performance may
  differ from the paper's model. The paper's model card lists MIMIC-III/IV, PTB-XL, Sleep-EDF and WAVES.
- **Benchmark.** The paper does not state the CinC 2012 windows, context and horizon lengths, normalization or split; its
  appendix lists horizons of 24-64 steps for the pre-training corpus. We use the 36 h → next 3 observation times
  benchmark with the APN split, so the numbers are not directly comparable.
- **Scale of the paper's numbers.** Table 2 labels the CinC 2012 row "(10¹)" without saying whether values are
  multiplied or divided by 10, and in which units (raw or normalized). If the values are ×10, MIRA-Large's RMSE is 0.62,
  the same order as the z-score RMSEs here.
- **Normalization.** We standardize each series with the mean and standard deviation of its history only.
  `model_eval.py` in the MIRA repo uses the whole window, including the values being forecast.
- **ODE block.** The authors' inference example skips it; we report both.

## 8. Notes for the authors: paper vs. released code

- **CinC 2012 metrics.** In Table 2, RMSE and MAE are nearly equal for every model on CinC 2012 (e.g. 5.987 / 5.985). That
  is what per-window averaging gives when each window has one forecast point (RMSE = |error| = MAE). Could you confirm the
  CinC 2012 horizon and how the metrics are averaged?
- **ODE time units.** In `MIRAForPrediction.forward`, the transformer sees the times rescaled by
  `normalize_time_for_ctrope` (to `[0, L-1]`), but the ODE block integrates over `next_target_time_values - time_values[:, -1]`
  in the raw input units. So the ODE horizon depends on the unit of the input times, while the transformer does not.
  Which scale was used in pre-training (`TimeAwareWindowDataset` passes `time_normalizer`-scaled times)?
- **Time rescaling at inference.** `model_eval.py` and the README rescale the times before calling the model, and the model
  rescales them again in every forward call, each time to `[0, L-1]` for the current length L. So CT-RoPE sees the
  relative spacing within the window, not absolute gaps, and the positions of all earlier points change after each
  autoregressive step. A KV cache built in one step therefore holds keys with outdated positions in the next one
  (PyHealth re-encodes the series at every step instead).
- **Leakage in `model_eval.py`.** Per-series mean and standard deviation are computed over context + horizon.
- **KV cache.** `MIRAAttention.forward` calls `past_key_value.get_usable_length` before checking for `None`, so
  `use_cache=False` raises; with a non-empty cache, the past length is added to `kv_seq_len` twice.
- **Port check.** PyHealth's `MIRABackbone` loads the checkpoint with all 453,331,072 parameters matched and reproduces
  `MIRAForPrediction` (with `torchdiffeq`) to within 1e-6 on random series, with and without the ODE block.